# Locket Van Gogh eval on Colab

Runs the photo-set comparison: the config before k3 (`van_gogh_before_k3.yaml`) against the current
`van_gogh.yaml`, on the same engine, same seed. Runtime -> **Change runtime type -> T4 GPU**, then **Run all**.

One-time, in Google Drive `MyDrive/locket/`:
- `server.tar` made by `python colab/pack_code.py` (it contains the eval code too);
- a folder `eval_photos/` with the .jpg files of `server/eval/photos` (the owner's own photos
  are private: they go to your Google Drive, nowhere else).

Results (images, `results.json`, comparison sheets) are written to `MyDrive/locket/eval_out/` and survive the
session: if Colab stops, **Run all** again and it resumes. Roughly 30-60 minutes for 2 x 62 photos
(an estimate, not measured on Colab yet). The last cell prints the tables and shows the sheets.

In [ ]:
import subprocess
import urllib.request

gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu or "NO GPU -> Runtime -> Change runtime type -> T4 GPU, then run again")

In [ ]:
import os
import pathlib
import sys
import tarfile

from google.colab import drive

drive.mount("/content/drive")
DRIVE = pathlib.Path("/content/drive/MyDrive/locket")
DRIVE.mkdir(exist_ok=True)
WORK = pathlib.Path("/content")

assert (DRIVE / "server.tar").exists(), "Upload server.tar to Google Drive -> MyDrive/locket/ (see the first cell)"
tarfile.open(DRIVE / "server.tar").extractall(WORK, filter="data")      # always the latest code from Drive
print("code ready")

In [ ]:
# Colab runs Python 3.13, but the pinned libraries (numpy 1.26, mediapipe 0.10.21) stop at 3.12, so the
# server gets its own Python 3.12 environment with the same pins as server/requirements.txt.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
VENV = WORK / "venv"
PY = str(VENV / "bin" / "python")
if not VENV.exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.12", str(VENV)], check=True)
skip = ("torch", "torchvision", "pytest", "httpx", "open_clip_torch")
pins = ["torch==2.6.0", "torchvision==0.21.0"]
for line in (WORK / "server" / "requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if line and not line.lower().startswith(skip):
        pins.append(line)
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", PY, *pins], check=True)
print("libraries ready")

In [ ]:
# Metric libraries the server does not need (CLIP style score, face-embedding similarity).
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", PY,
                "ftfy", "regex", "timm", "tqdm", "requests"], check=True)
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", PY, "--no-deps",
                "open_clip_torch==3.3.0", "facenet-pytorch"], check=True)
print("metric libraries ready")

In [ ]:
MODELS = WORK / "models"
WEIGHTS = {
    "RealESRGAN_x2plus.pth":
        "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",
    "blaze_face_short_range.tflite":
        "https://storage.googleapis.com/mediapipe-models/face_detector/"
        "blaze_face_short_range/float16/1/blaze_face_short_range.tflite",
}

if not MODELS.exists():
    if (DRIVE / "models.tar").exists():
        print("restoring models from Drive (~2 min)...")
        MODELS.mkdir()
        tarfile.open(DRIVE / "models.tar").extractall(MODELS, filter="data")
    else:
        print("first run: downloading ~6 GB of models (~5-10 min), saved to Drive for next time")
        (MODELS / "weights").mkdir(parents=True)
        for name, url in WEIGHTS.items():
            urllib.request.urlretrieve(url, MODELS / "weights" / name)
        env = dict(os.environ, HF_HOME=str(MODELS / "hf"), TORCH_HOME=str(MODELS / "torch"))
        subprocess.run([PY, "download_models.py"], cwd=WORK / "server", env=env, check=True)
        with tarfile.open(DRIVE / "models.tar", "w") as tar:
            tar.add(MODELS / "hf", arcname="hf")
            tar.add(MODELS / "weights", arcname="weights")
if not (WORK / "server" / "weights").exists():
    os.symlink(MODELS / "weights", WORK / "server" / "weights")
print("models ready")

In [ ]:
import collections
import shutil

# The folder name does not matter: use the folder under MyDrive/locket that holds the most .jpg files.
folders = collections.Counter(p.parent for p in DRIVE.rglob("*") if p.suffix.lower() == ".jpg")
assert folders, ("No .jpg file found under MyDrive/locket/. Upload the .jpg files of server/eval/photos "
                 "there (any folder name). If you just uploaded or renamed them, run the Drive cell "
                 "again. Colab sees: " + str(sorted(p.name for p in DRIVE.iterdir())))
PHOTOS_DRIVE, photo_count = folders.most_common(1)[0]
print("photos folder:", PHOTOS_DRIVE, "->", photo_count, ".jpg files")
EVAL = WORK / "server" / "eval"
if (EVAL / "photos").exists():
    shutil.rmtree(EVAL / "photos")
shutil.copytree(PHOTOS_DRIVE, EVAL / "photos")        # local copy: reading Drive per photo is slow
OUT_DRIVE = DRIVE / "eval_out"
OUT_DRIVE.mkdir(exist_ok=True)
out = EVAL / "out"
if out.is_symlink():
    out.unlink()
elif out.exists():
    shutil.rmtree(out)
os.symlink(OUT_DRIVE, out)                            # results live in Drive: a rerun resumes
print(len(list((EVAL / "photos").glob("*.jpg"))), "photos; already in eval_out:",
      sorted(p.name for p in OUT_DRIVE.iterdir()))

In [ ]:
import collections

ENV = dict(os.environ, HF_HOME=str(MODELS / "hf"), TORCH_HOME=str(MODELS / "torch"),
           PYTHONWARNINGS="ignore", PYTHONUNBUFFERED="1", MPLBACKEND="Agg")  # Colab's own backend breaks mediapipe

# "name:config file" = that whole config; a bare name = overrides from eval/sweep.py on van_gogh.yaml.
# new = k3 (today's yaml, brush relief off); imp1 = + brush relief; fast1 = fewer steps; imp1_fast1 = both.
VARIANTS = ["new:van_gogh.yaml", "imp1", "fast1", "imp1_fast1"]
# None = all photos. For a quick first look, e.g.:
# SUBSET = ["photo_02", "photo_08", "photo_12", "photo_17", "photo_31"]
SUBSET = None

env = dict(ENV)
if SUBSET:
    subset_dir = EVAL / "photos_subset"
    shutil.rmtree(subset_dir, ignore_errors=True)
    subset_dir.mkdir()
    for name in SUBSET:
        shutil.copy(EVAL / "photos" / f"{name}.jpg", subset_dir)
    env["SWEEP_PHOTOS"] = str(subset_dir)

SHOW = ("[", "SUMMARY", "Traceback") + tuple(v.split(":")[0] + " " for v in VARIANTS)


def run_stream(cmd):
    proc = subprocess.Popen(cmd, cwd=WORK / "server", env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    tail = collections.deque(maxlen=40)
    for line in proc.stdout:
        tail.append(line.rstrip()[:200])
        if line.startswith(SHOW) or "FAILED" in line:
            print(line.rstrip()[:180])
    code = proc.wait()
    if any(line.startswith("Traceback") for line in tail):
        print("--- last lines of the output (a chunk crashed) ---")
        print(chr(10).join(tail))
    return code


print("exit code", run_stream([PY, "-m", "eval.sweep_chunked", *VARIANTS]))

In [ ]:
from IPython.display import Image, display

for other in [v.split(":")[0] for v in VARIANTS][1:]:
    print("=" * 30, "new vs", other)
    done = subprocess.run([PY, "-m", "eval.summary", "new", other, "--sheets", "2"], cwd=WORK / "server",
                          env=ENV, capture_output=True, text=True)
    print(done.stdout)
    if done.returncode:
        print(done.stderr[-2000:])
    for sheet in sorted((EVAL / "out").glob(f"compare_{other}_*.png")):
        print(sheet.name)
        display(Image(str(sheet)))